# Exp 28: JPEG quality 10, non-VIP vs all

Card: [exp28_jpeg.md](../../docs/neuroslope/experiments/exp28_jpeg.md)


In [ ]:
import matplotlib.pyplot as plt
import albumentations as A
import numpy as np
from PIL import Image

from training.config import ROOT, TRAIN_DIR
from training.config import VIP
from training.cv import CVResult
from training.runs import read_config
from dataset.dataset import read_train
from analysis.gradcam import GradCamFigures
from analysis.plots import pr_curves, save_figures, use_style

N_SAMPLE = 16  # exp19 sample
N_PREVIEW = 6
N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
QUALITIES = [None, 10]  # None: original
CONFIG_DIR = ROOT / "configs" / "models"
CONFIGS = {
    f"JPEG {rows}": CONFIG_DIR / "exp28_jpeg" / f"resnet18_{rows.replace('-', '_').lower()}.yaml"
    for rows in ("non-VIP", "all")
}
EXP20 = CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.yaml"
EXP16 = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp28_jpeg"

# dataviz reference palette, light mode
SERIES = {"exp20": "#2a78d6", "JPEG non-VIP": "#1baf7a", "JPEG all": "#eb6834"}
use_style()

## JPEG compression examples


In [ ]:
train = read_train()
generic = train[train[VIP].sum(axis=1).eq(0) & train[["Cat", "Dog"]].sum(axis=1).gt(0)]
ids = generic.sample(N_PREVIEW, random_state=SEED).image_id
fig, axes = plt.subplots(
    len(ids),
    len(QUALITIES),
    figsize=(2.6 * len(QUALITIES), 2.8 * len(ids)),
    squeeze=False,
)
for r, image_id in enumerate(ids):
    image = A.Resize(read_config(EXP20)["data"]["img_size"], read_config(EXP20)["data"]["img_size"])(
        image=np.array(Image.open(TRAIN_DIR / f"{image_id}.jpg").convert("RGB"))
    )["image"]
    for k, q in enumerate(QUALITIES):
        shown = (
            image
            if q is None
            else A.ImageCompression(quality_range=(q, q), p=1)(image=image)["image"]
        )
        axes[r, k].imshow(shown)
        axes[r, k].set_title("original" if q is None else f"quality {q}")
for ax in axes.flat:
    ax.set_axis_off()
fig.tight_layout()
figures = {f"{PREFIX}_examples.png": fig}
plt.show()

## Training


In [ ]:
results = {"exp20": CVResult.load(EXP20)}
results.update({name: CVResult.load(cfg) for name, cfg in CONFIGS.items()})

## Grad-CAM Cat, Dog: exp20 vs JPEG non-VIP vs JPEG all


In [ ]:
cams = {"exp20": GradCamFigures(EXP20)} | {
    name: GradCamFigures(cfg) for name, cfg in CONFIGS.items()
}
exp16 = GradCamFigures(EXP16)
for cls in ["Cat", "Dog"]:
    # exp19 rows; smoke OOF holds few rows
    rows = exp16.positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM Cat, Dog: trained images


In [ ]:
for name in CONFIGS:
    slug = name.removeprefix("JPEG ").replace("-", "_").lower()
    for cls in ["Cat", "Dog"]:
        # exp19 sample: same seed over same out-of-fold rows
        fig = cams[name].positives(cls, train=True, n=N_SAMPLE, seed=SEED)
        figures[f"{PREFIX}_{slug}_grad_cam_{cls.lower()}_train.png"] = fig
plt.show()

## Grad-CAM VIP: unseen vs trained images


In [ ]:
for name in CONFIGS:
    slug = name.removeprefix("JPEG ").replace("-", "_").lower()
    for cls in VIP:
        for train in (False, True):
            fig = cams[name].positives(cls, train=train, n=N_POSITIVES)
            split = "train" if train else "validation"
            figures[f"{PREFIX}_{slug}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

## Precision-recall curves


In [ ]:
fig = pr_curves(results, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)